In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copy the zip from Drive to the fast local Colab storage
!cp "/content/drive/MyDrive/ml_dataset_hd.zip" /content/

In [ ]:
# Unzip it silently (-q) so it doesn't crash your browser with 4,000 print lines
!unzip -q /content/ml_dataset_hd.zip -d /content/

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18

# ================= 1. CONFIGURATION =================
# Point to the local, unzipped folder in Colab
DATASET_DIR = "/content/VoxEffects_ml_dataset"
BATCH_SIZE = 32
EPOCHS = 25                  # Reduced from 50
LEARNING_RATE = 3e-4         # Lowered to stabilize training

# Automatically target the Colab GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} for training.")
# ====================================================

# ================= 2. DATASET CLASS =================
class VoxEffectsDataset(Dataset):
    def __init__(self, dataset_dir, split="train"):
        self.folder_path = os.path.join(dataset_dir, split)
        self.file_paths = glob.glob(os.path.join(self.folder_path, "*.pt"))
        if len(self.file_paths) == 0:
            print(f"Warning: No .pt files found in {self.folder_path}")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        bundle = torch.load(self.file_paths[idx], map_location='cpu')
        return bundle['spectrogram'], bundle['targets']

# --- STRATEGY 1: SpecAugment ---
def apply_spec_augment(spectrograms):
    """Randomly masks out frequency and time bands to force the network to look at the whole picture."""
    batch_size, channels, freqs, frames = spectrograms.size()
    aug_spectrograms = spectrograms.clone()

    # Max mask parameters updated for 513-bin HD Linear Spectrograms
    max_f_mask = 64  # Mask up to ~12% of the 513 frequency bins
    max_t_mask = 80  # Mask up to ~5% of the 1723 time frames

    for i in range(batch_size):
        # Frequency Masking
        f = torch.randint(0, max_f_mask, (1,)).item()
        f0 = torch.randint(0, freqs - f, (1,)).item()
        aug_spectrograms[i, 0, f0:f0+f, :] = aug_spectrograms.mean()

        # Time Masking
        t = torch.randint(0, max_t_mask, (1,)).item()
        t0 = torch.randint(0, frames - t, (1,)).item()
        aug_spectrograms[i, 0, :, t0:t0+t] = aug_spectrograms.mean()

    return aug_spectrograms

# ================= 3. MODEL ARCHITECTURE =================
class VoxEffectsNet(nn.Module):
    def __init__(self):
        super(VoxEffectsNet, self).__init__()
        self.backbone = resnet18(weights=None)

        self.backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
        )
        self.backbone.fc = nn.Identity()

        # --- STRATEGY 1: Dropout Layer ---
        self.dropout = nn.Dropout(p=0.5)

        # 5 DSP heads (Limiter permanently removed)
        self.head_denoise = nn.Linear(512, 3)
        self.head_compressor = nn.Linear(512, 5)
        self.head_eq = nn.Linear(512, 7)
        self.head_deesser = nn.Linear(512, 3)
        self.head_reverb = nn.Linear(512, 4)

    def forward(self, x):
        shared_embedding = self.backbone(x)

        # Apply Dropout before feeding the heads
        shared_embedding = self.dropout(shared_embedding)

        return {
            'denoise': self.head_denoise(shared_embedding),
            'compressor': self.head_compressor(shared_embedding),
            'eq': self.head_eq(shared_embedding),
            'deesser': self.head_deesser(shared_embedding),
            'reverb': self.head_reverb(shared_embedding)
        }

# ================= 4. TRAINING LOOP =================
def train_model():
    train_dataset = VoxEffectsDataset(DATASET_DIR, split="train")
    val_dataset = VoxEffectsDataset(DATASET_DIR, split="val")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = VoxEffectsNet().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    criterion = nn.KLDivLoss(reduction='batchmean')
    modules = ['denoise', 'compressor', 'eq', 'deesser', 'reverb']

    # --- STRATEGY 1: Checkpoint Tracking Variable ---
    best_exact_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs = inputs.to(device)

            # Dynamically apply SpecAugment only to the training batch
            inputs = apply_spec_augment(inputs)

            targets = {k: v.to(device) for k, v in targets.items()}

            optimizer.zero_grad()
            outputs = model(inputs)

            batch_loss = 0.0
            for mod in modules:
                log_probs = F.log_softmax(outputs[mod], dim=1)
                batch_loss += criterion(log_probs, targets[mod])

            batch_loss.backward()
            optimizer.step()
            train_loss += batch_loss.item()

        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        total_samples = 0

        correct_top1 = {mod: 0 for mod in modules}
        regret_soft = {mod: 0.0 for mod in modules}
        exact_preset_correct = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = {k: v.to(device) for k, v in targets.items()}

                outputs = model(inputs)
                batch_size_current = inputs.size(0)
                total_samples += batch_size_current

                batch_loss = 0.0
                exact_match_batch = torch.ones(batch_size_current, dtype=torch.bool, device=device)

                for mod in modules:
                    log_probs = F.log_softmax(outputs[mod], dim=1)
                    batch_loss += criterion(log_probs, targets[mod])

                    pred_class = torch.argmax(outputs[mod], dim=1)
                    target_class = torch.argmax(targets[mod], dim=1)

                    correct_top1[mod] += (pred_class == target_class).sum().item()
                    exact_match_batch &= (pred_class == target_class)

                    max_target_probs, _ = torch.max(targets[mod], dim=1)
                    pred_target_probs = targets[mod][torch.arange(batch_size_current), pred_class]
                    regret_soft[mod] += (max_target_probs - pred_target_probs).sum().item()

                val_loss += batch_loss.item()
                exact_preset_correct += exact_match_batch.sum().item()

        avg_val_loss = val_loss / len(val_loader)

        # --- LOGGING & CHECKPOINTING ---
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        exact_acc = (exact_preset_correct / total_samples) * 100
        print(f"Global Exact Preset Accuracy (All 5 Modules): {exact_acc:.2f}%")

        print("\nModule-Level Metrics:")
        for mod in modules:
            acc = (correct_top1[mod] / total_samples) * 100
            avg_regret = regret_soft[mod] / total_samples
            print(f"  - {mod.capitalize():>10} | Top-1: {acc:5.2f}% | Soft Regret: {avg_regret:.4f}")

       # --- STRATEGY 1: Best Accuracy Checkpointing ---
        if exact_acc > best_exact_acc:
            best_exact_acc = exact_acc
            # Save the new optimal weights
            torch.save(model.state_dict(), "/content/drive/MyDrive/voxeffects_model_best_acc.pth")
            print(f"\n🌟 NEW BEST MODEL! Saved to Google Drive. (Accuracy jumped to {best_exact_acc:.2f}%)")
        else:
            print(f"\nNo improvement. Best Accuracy remains {best_exact_acc:.2f}%")

    print("\nTraining Complete! Your optimal HD epoch weights are secured in Google Drive.")

if __name__ == "__main__":
    train_model()

Using device: cuda for training.

--- Epoch 1/25 ---
Train Loss: 0.8608 | Val Loss: 0.7933
Global Exact Preset Accuracy (All 5 Modules): 10.44%

Module-Level Metrics:
  -    Denoise | Top-1: 30.23% | Soft Regret: 0.0670
  - Compressor | Top-1: 58.45% | Soft Regret: 0.1937
  -         Eq | Top-1: 69.18% | Soft Regret: 0.1064
  -    Deesser | Top-1: 98.29% | Soft Regret: 0.0007
  -     Reverb | Top-1: 68.35% | Soft Regret: 0.0027

🌟 NEW BEST MODEL! Saved to Google Drive. (Accuracy jumped to 10.44%)

--- Epoch 2/25 ---
Train Loss: 0.6514 | Val Loss: 0.7531
Global Exact Preset Accuracy (All 5 Modules): 9.65%

Module-Level Metrics:
  -    Denoise | Top-1: 47.13% | Soft Regret: 0.0719
  - Compressor | Top-1: 53.06% | Soft Regret: 0.1945
  -         Eq | Top-1: 70.75% | Soft Regret: 0.0924
  -    Deesser | Top-1: 98.29% | Soft Regret: 0.0007
  -     Reverb | Top-1: 70.95% | Soft Regret: 0.0041

No improvement. Best Accuracy remains 10.44%

--- Epoch 3/25 ---
Train Loss: 0.5699 | Val Loss: 0.8

In [ ]:
import os
import glob
import torch

# ================= CONFIGURATION =================
# If running on the AUTH cluster:
#TARGETS_DIR = "/home/l/lialiosk/VoxEffects/training_targets"

# If running on Google Colab, you can point this directly to the unzipped slices:
TARGETS_DIR = "/content/VoxEffects_ml_dataset/train"
# =================================================

def audit_targets():
    # rglog searches recursively, finding standalone targets or ML data bundles
    target_files = glob.glob(os.path.join(TARGETS_DIR, "**", "*.pt"), recursive=True)

    if not target_files:
        print(f"No .pt files found in {TARGETS_DIR}. Please check the path.")
        return

    modules = ['denoise', 'compressor', 'eq', 'deesser', 'reverb', 'limiter']

    sums = {}
    top1_counts = {}
    total_files = 0

    for filepath in target_files:
        try:
            # CPU mapping ensures it runs safely anywhere without GPU RAM spikes
            data = torch.load(filepath, map_location='cpu')

            # Automatically detect if this is a standalone target or a full data bundle
            targets = data['targets'] if 'targets' in data else data

            # Initialize the tracking tensors on the first successful load
            if total_files == 0:
                for mod in modules:
                    num_classes = targets[mod].shape[0]
                    sums[mod] = torch.zeros(num_classes)
                    top1_counts[mod] = torch.zeros(num_classes)

            for mod in modules:
                probs = targets[mod]
                sums[mod] += probs

                # Track which option was the "winner" (argmax) for this specific file
                top1_idx = torch.argmax(probs).item()
                top1_counts[mod][top1_idx] += 1

            total_files += 1

        except Exception as e:
            continue

    if total_files == 0:
        print("No valid target tensors found to parse.")
        return

    print(f"--- Audit Complete: Analyzed {total_files} Target Tensors ---\n")

    for mod in modules:
        num_classes = sums[mod].shape[0]
        mean_probs = sums[mod] / total_files
        top1_percentages = (top1_counts[mod] / total_files) * 100

        print(f"MODULE: {mod.upper()} ({num_classes} options)")
        print(f"{'Option':<8} | {'Mean Probability':<18} | {'Top-1 Win Rate':<15}")
        print("-" * 49)
        for i in range(num_classes):
            prob = mean_probs[i].item()
            win_rate = top1_percentages[i].item()
            print(f"Opt {i:<4} | {prob:.4f}           | {win_rate:6.2f}%")
        print("\n")

if __name__ == "__main__":
    audit_targets()

--- Audit Complete: Analyzed 8437 Target Tensors ---

MODULE: DENOISE (3 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.3448           |  40.44%
Opt 1    | 0.3435           |  21.45%
Opt 2    | 0.3117           |  38.11%


MODULE: COMPRESSOR (5 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.2744           |  20.67%
Opt 1    | 0.0800           |   0.00%
Opt 2    | 0.0819           |   0.24%
Opt 3    | 0.2290           |  18.30%
Opt 4    | 0.3347           |  60.79%


MODULE: EQ (7 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.1543           |   2.61%
Opt 1    | 0.1558           |  18.54%
Opt 2    | 0.1659           |  22.92%
Opt 3    | 0.1663           |  15.10%
Opt 4    | 0.1436           |  37.68%
Opt 5    | 0.1543           |   0.00%
Opt 6    | 0.0600           |   3.

In [ ]:
import os
import glob
import torch

# ================= CONFIGURATION =================
# If running on the AUTH cluster:
#TARGETS_DIR = "/home/l/lialiosk/VoxEffects/training_targets"

# If running on Google Colab, you can point this directly to the unzipped slices:
TARGETS_DIR = "/content/VoxEffects_ml_dataset/val"
# =================================================

def audit_targets():
    # rglog searches recursively, finding standalone targets or ML data bundles
    target_files = glob.glob(os.path.join(TARGETS_DIR, "**", "*.pt"), recursive=True)

    if not target_files:
        print(f"No .pt files found in {TARGETS_DIR}. Please check the path.")
        return

    modules = ['denoise', 'compressor', 'eq', 'deesser', 'reverb', 'limiter']

    sums = {}
    top1_counts = {}
    total_files = 0

    for filepath in target_files:
        try:
            # CPU mapping ensures it runs safely anywhere without GPU RAM spikes
            data = torch.load(filepath, map_location='cpu')

            # Automatically detect if this is a standalone target or a full data bundle
            targets = data['targets'] if 'targets' in data else data

            # Initialize the tracking tensors on the first successful load
            if total_files == 0:
                for mod in modules:
                    num_classes = targets[mod].shape[0]
                    sums[mod] = torch.zeros(num_classes)
                    top1_counts[mod] = torch.zeros(num_classes)

            for mod in modules:
                probs = targets[mod]
                sums[mod] += probs

                # Track which option was the "winner" (argmax) for this specific file
                top1_idx = torch.argmax(probs).item()
                top1_counts[mod][top1_idx] += 1

            total_files += 1

        except Exception as e:
            continue

    if total_files == 0:
        print("No valid target tensors found to parse.")
        return

    print(f"--- Audit Complete: Analyzed {total_files} Target Tensors ---\n")

    for mod in modules:
        num_classes = sums[mod].shape[0]
        mean_probs = sums[mod] / total_files
        top1_percentages = (top1_counts[mod] / total_files) * 100

        print(f"MODULE: {mod.upper()} ({num_classes} options)")
        print(f"{'Option':<8} | {'Mean Probability':<18} | {'Top-1 Win Rate':<15}")
        print("-" * 49)
        for i in range(num_classes):
            prob = mean_probs[i].item()
            win_rate = top1_percentages[i].item()
            print(f"Opt {i:<4} | {prob:.4f}           | {win_rate:6.2f}%")
        print("\n")

if __name__ == "__main__":
    audit_targets()

--- Audit Complete: Analyzed 2041 Target Tensors ---

MODULE: DENOISE (3 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.3465           |  38.76%
Opt 1    | 0.3453           |  20.58%
Opt 2    | 0.3081           |  40.67%


MODULE: COMPRESSOR (5 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.2888           |  23.32%
Opt 1    | 0.0679           |   0.00%
Opt 2    | 0.0732           |   0.00%
Opt 3    | 0.2276           |  11.12%
Opt 4    | 0.3426           |  65.56%


MODULE: EQ (7 options)
Option   | Mean Probability   | Top-1 Win Rate 
-------------------------------------------------
Opt 0    | 0.1557           |   1.57%
Opt 1    | 0.1523           |   8.57%
Opt 2    | 0.1684           |  33.37%
Opt 3    | 0.1678           |  11.56%
Opt 4    | 0.1591           |  44.63%
Opt 5    | 0.1557           |   0.00%
Opt 6    | 0.0410           |   0.

In [ ]:
import torch
from torch.utils.mobile_optimizer import optimize_for_mobile

# 1. Initialize your architecture and load the winning weights
model = VoxEffectsNet()
# Make sure to map to CPU so it doesn't look for a GPU on the mobile phone
model.load_state_dict(torch.load("/content/drive/MyDrive/voxeffects_model_best_acc.pth", map_location='cpu'))
model.eval()

# 2. Create an Android Wrapper
# This translates the Python Dictionary into a static Tuple for Kotlin
class AndroidMobileWrapper(nn.Module):
    def __init__(self, core_model):
        super().__init__()
        self.core = core_model

    def forward(self, x):
        out = self.core(x)
        # Order matters! Kotlin will read these via index [0], [1], [2], [3], [4]
        return out['denoise'], out['compressor'], out['eq'], out['deesser'], out['reverb']

# Wrap the loaded model
mobile_model = AndroidMobileWrapper(model)
mobile_model.eval()

# 3. Create dummy input [Batch, Channels, Freq Bins, Time Frames]
dummy_input = torch.rand(1, 1, 513, 1723)

# 4. Trace the WRAPPED model (No strict=False needed anymore)
traced_script_module = torch.jit.trace(mobile_model, dummy_input)

# 5. Optimize and save for PyTorch Lite
optimized_traced_model = optimize_for_mobile(traced_script_module)
optimized_traced_model._save_for_lite_interpreter("/content/drive/MyDrive/voxeffects_mobile.ptl")
print("Saved Android-ready model as voxeffects_mobile.ptl")

Saved Android-ready model as voxeffects_mobile.ptl


/tmp/ipykernel_577/1762803888.py:34: DeprecationWarning: Lite Interpreter is deprecated. Please consider switching to ExecuTorch.             https://docs.pytorch.org/executorch/stable/getting-started.html
  optimized_traced_model._save_for_lite_interpreter("/content/drive/MyDrive/voxeffects_mobile.ptl")
